# GRAMSCI quickstart

Measure the **2-, 3- and 4-point correlation functions** of a 3D catalogue
from Python, in a few lines.

This notebook uses the small mock catalogue shipped in `example/` (`test.gal`,
`test.ran`). It runs in a few seconds.

**Before you start:** build the CPU binary once from the repo root with `make`
(produces `bin/gramsci`). The Python package finds it automatically.

In [ ]:
import sys, pathlib
# Use the in-repo package without installing it (or run: pip install -e ../python)
sys.path.insert(0, str(pathlib.Path.cwd().parent / "python"))

import numpy as np
import matplotlib.pyplot as plt
import gramsci

print("gramsci", gramsci.__version__)

## 1. Load a catalogue

GRAMSCI works on a data catalogue plus a random catalogue covering the same
volume (the randoms set the survey geometry for the RR normalisation). Columns
are `x y z weight` in comoving Mpc/h. Here we subsample for speed.

In [ ]:
gal = np.loadtxt("test.gal")   # columns: x y z weight
ran = np.loadtxt("test.ran")

rng = np.random.default_rng(0)
gi = rng.choice(len(gal), 30000, replace=False)
ri = rng.choice(len(ran), 60000, replace=False)
data, wd = gal[gi, :3], gal[gi, 3]
rand, wr = ran[ri, :3], ran[ri, 3]
print("data:", data.shape, "  randoms:", rand.shape)

## 2. Two-point correlation function

`compute_2pcf` returns an object with `.r` (bin midpoints), `.xi`, and the raw
`.NN` / `.RR` counts.

In [ ]:
xi = gramsci.compute_2pcf(data, wd, rand, wr, rmin=1, rmax=80, nbins=20)

plt.figure(figsize=(6, 4))
plt.plot(xi.r, xi.r**2 * xi.xi, "o-")
plt.xlabel("r  [Mpc/h]")
plt.ylabel(r"$r^2\,\xi(r)$")
plt.title("2-point correlation function")
plt.tight_layout()
plt.show()

## 3. Three-point correlation function

`compute_3pcf` returns one row per triangle configuration `(r1, r2, r3)`, with
the connected estimate in `.zeta`. Below we pick out the **equilateral**
triangles (`r1 = r2 = r3`).

In [ ]:
z = gramsci.compute_3pcf(data, wd, rand, wr, rmin=1, rmax=50, nbins=6)
print(z, " -> arrays: z.r1, z.r2, z.r3, z.NNN, z.RRR, z.zeta\n")

eq = np.isclose(z.r1, z.r2) & np.isclose(z.r2, z.r3)
for r, zeta in zip(z.r1[eq], z.zeta[eq]):
    print(f"  equilateral r = {r:5.1f} Mpc/h   zeta = {zeta: .4g}")

## 4. Four-point correlation function (incl. parity)

`compute_4pcf` gives the connected 4PCF (`.zeta_conn`); with `parity=True` it
splits into parity-**even** (`.zeta_even`) and parity-**odd** (`.zeta_odd`)
channels — the odd channel is a probe of parity violation in the galaxy field.

In [ ]:
q  = gramsci.compute_4pcf(data, wd, rand, wr, rmin=1, rmax=30, nbins=3)
qp = gramsci.compute_4pcf(data, wd, rand, wr, rmin=1, rmax=30, nbins=3, parity=True)
print(q,  " -> q.zeta_conn")
print(qp, " -> qp.zeta_even, qp.zeta_odd")

## Running on the GPU

Build a GPU backend (`make gpu` for the portable OpenCL build, `make cuda` for
NVIDIA) and point any call at it:

```python
z = gramsci.compute_3pcf(data, wd, rand, wr, rmin=1, rmax=50, nbins=6,
                         binary="../bin/gramsci_cl")
```

(or `export GRAMSCI_BIN=.../bin/gramsci_cl`). Same results — see the repository
README and `src_opencl/README.md` for precision and performance notes.

## Next steps

- Scale up: use the full catalogue and a larger `rmax` (e.g. 150 Mpc/h) to
  resolve the **baryon acoustic feature** — see `example/bao_3pcf.png` and
  `example/plot_bao_3pcf.py`.
- For very large catalogues, `bin/domain_decomposition` splits the volume.
- If you use GRAMSCI, please cite Sabiu et al. 2019, ApJS 242, 29
  ([arXiv:1901.00296](https://arxiv.org/abs/1901.00296)).